# 🐖 09 | Evaluación armonizada YOLO11n-seg frente a Mask R-CNN

**RECURSO: GPU T4**. No entrena ni altera pesos, no cambia semillas, umbrales ni particiones originales.

**Objetivo:** recalcular métricas **COCOeval de máscaras** en las mismas **373 imágenes** de prueba congelada por fecha, utilizando **exactamente el mismo terreno de referencia (GT) y el mismo evaluador** para ambas arquitecturas. Se guardan predicciones en formato COCO RLE para reproducir análisis posteriores con CPU.

> **Integridad científica:** esta es una armonización de evaluación sobre un test *ya consultado* en notebooks 04/07. **No es un nuevo test independiente**. No ajustar hiperparámetros usando estos resultados. Diferencias de presupuesto de entrenamiento (YOLO 60 épocas, Mask R-CNN 12) siguen presentes.

**Secuencia:** ejecutar todas las celdas con GPU T4, autorizar Drive, recibir ZIP; después abrir notebook 10 con CPU.
**Corrección técnica (18-10-2026):** se evita la mutación de predicciones por `COCO.loadRes` al calcular AP global y por fecha. No se cambian el protocolo, los pesos, los umbrales ni los datos. Si los JSON de predicción existen en Drive, se reutilizan.


## 1. Comprobar la GPU antes de trabajar

Si Colab está con CPU, esta celda se detiene sin hacer ningún cómputo largo.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Activa GPU T4: Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU.'
print('✅ GPU:',torch.cuda.get_device_name(0))


In [ ]:
%pip -q install "ultralytics>=8.3,<9" "pycocotools>=2.0.8,<3" "pyyaml>=6,<7" "pandas>=2,<3" openpyxl pillow opencv-python-headless matplotlib


## 2. Montar Drive y bloquear datos/modelos usados anteriormente

Se comprueban los **SHA-256** de los pesos y el YAML frente a los informes de los notebooks anteriores. Si algo cambió, este módulo se detiene.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from contextlib import redirect_stdout
import hashlib,json,io,os,zipfile,platform,random,math,sys
import numpy as np,pandas as pd,yaml,cv2,torch,torchvision
from PIL import Image
from ultralytics import YOLO
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms.functional import to_tensor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as maskutils

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(), f'Proyecto no encontrado: {ROOT}'
DATA=ROOT/'01_Datasets'/'FaroPigSeg'
SPLIT=DATA/'split_fecha_v1'
RES=ROOT/'03_Resultados'
EXPORT=ROOT/'05_Exportaciones_ZIP'
RUN=RES/'06_Entrenamiento'
REPORTS=RES/'05_Reportes'
METRICS=RES/'03_Metricas'
FIGURES=RES/'04_Figuras'
for p in [EXPORT,REPORTS,METRICS,FIGURES]: p.mkdir(exist_ok=True,parents=True)
YAML=SPLIT/'data_fecha.yaml'
MANIFEST=SPLIT/'particion_fecha_manifest.json'
YOLO_W=RUN/'YOLO11n_fecha_v1'/'weights'/'best.pt'
MASK_W=RUN/'MaskRCNN_fecha_v1'/'maskrcnn_best_model.pt'
YOLO_OLD=METRICS/'CERDOS_04_Evaluacion_test.json'
MASK_OLD=REPORTS/'CERDOS_07_Evaluacion_MaskRCNN.json'
MASK_DONE=RUN/'MaskRCNN_fecha_v1'/'ENTRENAMIENTO_FINALIZADO.json'
for p in [YAML,MANIFEST,YOLO_W,MASK_W,YOLO_OLD,MASK_OLD,MASK_DONE]:
    if not p.is_file(): raise FileNotFoundError(f'Falta dependencia de notebooks 02–07: {p}')
def file_sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as handle:
        for chunk in iter(lambda:handle.read(4*1024*1024),b''):h.update(chunk)
    return h.hexdigest()
yolo_previous=json.loads(YOLO_OLD.read_text(encoding='utf8'))
mask_previous=json.loads(MASK_OLD.read_text(encoding='utf8'))
mask_done=json.loads(MASK_DONE.read_text(encoding='utf8'))
assert file_sha256(YOLO_W)==yolo_previous['weights_sha256'],'YOLO: los pesos cambiaron desde el notebook 04'
assert file_sha256(MASK_W)==mask_previous['weights_sha256'],'Mask R-CNN: los pesos cambiaron desde el notebook 07'
assert file_sha256(YAML)==mask_previous['yaml_sha256']==mask_done['yaml_sha256'],'El YAML cambió desde el entrenamiento/evaluación Mask R-CNN'
config=yaml.safe_load(YAML.read_text(encoding='utf8'))
paths=[Path(v.strip()) for v in Path(config['test']).read_text(encoding='utf8').splitlines() if v.strip()]
assert len(paths)==373, f'Se esperaban 373 imágenes de prueba y se encontraron {len(paths)}'
assert len({str(p) for p in paths})==373 and all(p.is_file() for p in paths),'Imágenes ausentes o repetidas'
parts=json.loads(MANIFEST.read_text(encoding='utf8'))['asignaciones_fecha']
assert all(parts.get(p.name[:10])=='test' for p in paths),'El test no coincide con el manifiesto por fecha'
assert len({p.name[:10] for p in paths})==5,'Número inesperado de fechas de prueba'
print('✅ Test congelado:',len(paths),'imágenes,',len({p.name[:10] for p in paths}),'fechas')
print('✅ SHA-256 pesos y YAML comprobados, no se realizará entrenamiento.')


## 3. Construcción ÚNICA de anotaciones COCO desde los polígonos originales

Las dos arquitecturas tendrán el **mismo archivo JSON de referencia**, construido con el procedimiento de máscaras poligonales usado en la evaluación anterior de Mask R-CNN. No se extraerán anotaciones de las predicciones.

In [ ]:
DIR=REPORTS/'CERDOS_09_COCO_Armonizado'
DIR.mkdir(exist_ok=True,parents=True)
GT_FILE=DIR/'CERDOS_09_GT_test_coco.json'
YPRED_FILE=DIR/'CERDOS_09_Pred_YOLO_coco.json'
MPRED_FILE=DIR/'CERDOS_09_Pred_MaskRCNN_coco.json'
REP_FILE=REPORTS/'CERDOS_09_Armonizacion_COCO.json'
EXCEL=METRICS/'CERDOS_09_Comparacion_Armonizada_COCO.xlsx'

PROTO={'pipeline':'CERDOS_09_COCOv1', 'image_count':373,'n_dates':5,'max_dets':100,
       'yolo_imgsz':640,'yolo_conf':0.001,'yolo_iou_nms':0.7,'yolo_retina_masks':True,
       'mask_score_thresh':0.001,'mask_bin_thresh':0.5,'gt_raster':'opencv_fillPoly_round_normalized',
       'test_file_sha256':file_sha256(Path(config['test'])),
       'yaml_sha256':file_sha256(YAML),'yolo_sha256':file_sha256(YOLO_W),
       'mask_sha256':file_sha256(MASK_W),'evaluator':'pycocotools COCOeval segm IoU .50:.95'}


# Proteger la reanudación: los JSON existentes solo se aceptan bajo el mismo protocolo.
CONFIG_FILE=DIR/'CERDOS_09_Protocolo.json'
if CONFIG_FILE.exists():
    previous=json.loads(CONFIG_FILE.read_text(encoding='utf8'))
    if previous!=PROTO:
        raise RuntimeError('La configuración 09 cambió. No reutilizar predicciones de otra versión.')
else:
    if any(p.exists() for p in [GT_FILE,YPRED_FILE,MPRED_FILE,REP_FILE,EXCEL]):
        raise RuntimeError('Hay resultados anteriores sin archivo de protocolo 09. Revisar manualmente antes de continuar.')
    CONFIG_FILE.write_text(json.dumps(PROTO,ensure_ascii=False,indent=2),encoding='utf8')


def encoded(mask):
    obj=maskutils.encode(np.asfortranarray(mask.astype(np.uint8)))
    obj['counts']=obj['counts'].decode('ascii')
    return obj

def make_gt():
    ims=[];anns=[];n=0
    for im_id,path in enumerate(paths,start=1):
        with Image.open(path) as image:w,h=image.size
        ims.append({'id':im_id,'width':w,'height':h,'file_name':path.name})
        lab=Path(str(path).replace('/images/','/labels/')).with_suffix('.txt')
        assert lab.is_file(),f'Falta etiqueta: {lab}'
        for row in lab.read_text(encoding='utf-8-sig').splitlines():
            if not row.strip():continue
            words=row.split()
            assert int(words[0])==0,'Clase distinta de cerdo'
            data=np.asarray([float(x) for x in words[1:]],dtype=np.float32)
            assert len(data)>=6 and len(data)%2==0 and np.isfinite(data).all() and np.all((data>=0)&(data<=1)),f'Polígono inválido: {lab}'
            coords=np.rint(data.reshape(-1,2)*[w,h]).astype(np.int32)
            coords[:,0]=np.clip(coords[:,0],0,w-1);coords[:,1]=np.clip(coords[:,1],0,h-1)
            mask=np.zeros((h,w),np.uint8)
            cv2.fillPoly(mask,[coords],1)
            if mask.sum()<3:raise ValueError(f'Máscara degenerada en {lab}')
            n+=1
            rle=encoded(mask)
            bbox=[float(x) for x in maskutils.toBbox(rle)]
            anns.append({'id':n,'image_id':im_id,'category_id':1,'segmentation':rle,
                         'bbox':bbox,'area':int(mask.sum()),'iscrowd':0})
        if im_id%100==0:print('GT procesado',im_id,'/',len(paths))
    if len(anns)!=4648:raise RuntimeError(f'Instancias GT inesperadas: {len(anns)}, se esperaban 4648')
    return {'info':{'description':'FaroPigSeg — test congelado por fecha'},'licenses':[],
            'images':ims,'annotations':anns,'categories':[{'id':1,'name':'cerdo','supercategory':'animal'}]}

if GT_FILE.exists():
    gt_json=json.loads(GT_FILE.read_text(encoding='utf8'))
    assert len(gt_json['images'])==373 and len(gt_json['annotations'])==4648,'GT previo incompleto'
else:
    gt_json=make_gt()
    GT_FILE.write_text(json.dumps(gt_json,ensure_ascii=False),encoding='utf8')
print('✅ Referencia COCO común:',len(gt_json['images']),'imágenes y',len(gt_json['annotations']),'instancias')


## 4. Inferencia YOLO11n-seg (sin entrenamiento)

Predice sobre las 373 imágenes una sola vez con la configuración registrada; las máscaras se mantienen en la resolución original (`retina_masks=True`). Si se interrumpe Colab, al volver a ejecutar se reutiliza la predicción terminada.

In [ ]:
if YPRED_FILE.is_file():
    ypred=json.loads(YPRED_FILE.read_text(encoding='utf8'))
    print('✅ Predicciones YOLO anteriores reutilizadas:',len(ypred))
else:
    detector=YOLO(str(YOLO_W))
    ypred=[]
    for i,p in enumerate(paths,start=1):
        result=detector.predict(source=str(p),device=0,imgsz=PROTO['yolo_imgsz'],
            conf=PROTO['yolo_conf'],iou=PROTO['yolo_iou_nms'],max_det=PROTO['max_dets'],
            retina_masks=True,verbose=False,save=False)[0]
        scores=result.boxes.conf.detach().cpu().numpy() if result.boxes is not None else np.empty(0)
        with Image.open(p) as image:ow,oh=image.size
        if len(scores):
            assert result.masks is not None and result.masks.data is not None,'Detecciones YOLO sin máscaras'
            seg=result.masks.data.detach().cpu().numpy()
            assert seg.shape==(len(scores),oh,ow),f'Máscaras YOLO {seg.shape}; se esperaba {(len(scores),oh,ow)}. Revisar retina_masks en vez de redimensionar a ciegas.'
            for s,m in zip(scores,seg):
                m=(m>0.5).astype(np.uint8)
                if m.any():ypred.append({'image_id':i,'category_id':1,'segmentation':encoded(m),'score':float(s)})
        if i%50==0:print('YOLO inferencia',i,'/',len(paths))
    YPRED_FILE.write_text(json.dumps(ypred,ensure_ascii=False),encoding='utf8')
    del detector;torch.cuda.empty_cache()
print('✅ YOLO instancias predichas para COCO:',len(ypred))


## 5. Inferencia Mask R-CNN con el mismo protocolo de evaluación

Se carga **best_model** (época 8 del entrenamiento original), sin iniciar un nuevo proceso de ajuste. Máscaras umbralizadas a 0,5 como en notebook 07.

In [ ]:
def build_maskrcnn():
    m=maskrcnn_resnet50_fpn(weights=None,weights_backbone=None,
        min_size=512,max_size=768,trainable_backbone_layers=3)
    dim=m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor=FastRCNNPredictor(dim,2)
    cin=m.roi_heads.mask_predictor.conv5_mask.in_channels
    m.roi_heads.mask_predictor=MaskRCNNPredictor(cin,256,2)
    m.roi_heads.score_thresh=PROTO['mask_score_thresh']
    m.roi_heads.detections_per_img=PROTO['max_dets']
    return m

if MPRED_FILE.is_file():
    mpred=json.loads(MPRED_FILE.read_text(encoding='utf8'))
    print('✅ Predicciones Mask R-CNN anteriores reutilizadas:',len(mpred))
else:
    model=build_maskrcnn().to('cuda:0')
    state=torch.load(MASK_W,map_location='cpu',weights_only=False)
    model.load_state_dict(state['model'],strict=True)
    del state
    model.eval(); mpred=[]
    for i,p in enumerate(paths,start=1):
        with Image.open(p) as image:rgb=np.asarray(image.convert('RGB')).copy()
        h,w=rgb.shape[:2]
        with torch.inference_mode():ret=model([to_tensor(rgb).to('cuda:0')])[0]
        scores=ret['scores'].detach().cpu().numpy()
        if len(scores):
            masks=ret['masks'].detach().cpu().numpy()[:,0]
            assert masks.shape==(len(scores),h,w),f'Máscaras Mask R-CNN inesperadas: {masks.shape}'
            for s,m in zip(scores,masks):
                binary=(m>PROTO['mask_bin_thresh']).astype(np.uint8)
                if binary.any():mpred.append({'image_id':i,'category_id':1,
                    'segmentation':encoded(binary),'score':float(s)})
        if i%50==0:print('Mask R-CNN inferencia',i,'/',len(paths))
    MPRED_FILE.write_text(json.dumps(mpred,ensure_ascii=False),encoding='utf8')
    del model;torch.cuda.empty_cache()
print('✅ Mask R-CNN instancias predichas para COCO:',len(mpred))


## 6. Evaluar ambos modelos con **el mismo COCOeval**, global y por fecha

La métrica es AP de **máscaras**, no detección de cajas. Se mantienen iguales las categorías, los IDs de imágenes, el terreno de referencia, los umbrales IoU y `maxDets=100`. Los AP por fecha no son independientes de las métricas globales.

In [ ]:
def as_coco_groundtruth(d):
    gt=COCO();gt.dataset=d
    with redirect_stdout(io.StringIO()):gt.createIndex()
    return gt

def as_coco_predictions(gt,rows):
    # COCO.loadRes modifica las anotaciones recibidas: puede insertar 'bbox'
    # como np.ndarray. Las mismas filas se reutilizan en la evaluación por fecha.
    # Crear una copia limpia en cada llamada evita comparar dicho array con [].
    if rows:
        safe_rows=[
            {'image_id':int(r['image_id']),
             'category_id':int(r['category_id']),
             'segmentation':r['segmentation'],
             'score':float(r['score'])}
            for r in rows
        ]
        with redirect_stdout(io.StringIO()):
            return gt.loadRes(safe_rows)
    dt=COCO();dt.dataset={'images':gt.dataset['images'],'categories':gt.dataset['categories'],'annotations':[]}
    with redirect_stdout(io.StringIO()):dt.createIndex()
    return dt

def compute_ap(gt,rows,img_ids=None):
    dt=as_coco_predictions(gt,rows)
    evaluator=COCOeval(gt,dt,'segm')
    evaluator.params.maxDets=[1,10,PROTO['max_dets']]
    if img_ids is not None:evaluator.params.imgIds=sorted(int(z) for z in img_ids)
    with redirect_stdout(io.StringIO()):
        evaluator.evaluate();evaluator.accumulate();evaluator.summarize()
    return {'mAP50_95_mascaras':float(evaluator.stats[0]),
            'mAP50_mascaras':float(evaluator.stats[1]),
            'mAP75_mascaras':float(evaluator.stats[2])}

if REP_FILE.exists():
    report=json.loads(REP_FILE.read_text(encoding='utf8'))
    if report.get('protocolo')!=PROTO:raise RuntimeError('Reporte 09 existente tiene otro protocolo o pesos. No sobrescribir.')
    if not EXCEL.is_file():raise FileNotFoundError('Reporte armonizado existente sin Excel: revisar Drive antes de ejecutar.')
    print('✅ Evaluación 09 armonizada ya existe; NO se repite.');display(pd.DataFrame(report['global']))
else:
    gt=as_coco_groundtruth(gt_json)
    grouped={}
    for i,p in enumerate(paths,start=1):grouped.setdefault(p.name[:10],[]).append(i)
    global_rows=[];date_rows=[]
    for name,rows in [('YOLO11n-seg',ypred),('Mask R-CNN ResNet50-FPN',mpred)]:
        g=compute_ap(gt,rows)
        global_rows.append({'modelo':name,'n_imagenes':len(paths),'gt_instancias':len(gt_json['annotations']),**g})
        for dt,ids in sorted(grouped.items()):
            metrics=compute_ap(gt,rows,ids)
            date_rows.append({'modelo':name,'fecha':dt,'n_imagenes':len(ids),**metrics})
        print('✅ Evaluación armonizada:',name,g)
    df_g=pd.DataFrame(global_rows)
    df_d=pd.DataFrame(date_rows)
    dif={key:float(df_g.iloc[0][key]-df_g.iloc[1][key]) for key in ['mAP50_95_mascaras','mAP50_mascaras','mAP75_mascaras']}
    report={'etapa':'09_evaluacion_armonizada_sin_entrenamiento','fecha_utc':datetime.now(timezone.utc).isoformat(),
        'protocolo':PROTO,'global':global_rows,'por_fecha':date_rows,
        'diferencia_YOLO_menos_MaskRCNN':dif,
        'notas':['Se reevaluó el mismo test ya consultado en 04 y 07; no es un test externo nuevo.',
        'Evaluador armonizado pero persistente desigualdad de épocas, parámetros y costo de entrenamiento.',
        'Una fecha de captura no equivale a una granja o animal independiente.',
        'No se modificaron pesos ni umbrales para elegir el mejor modelo.']}
    with pd.ExcelWriter(EXCEL,engine='openpyxl') as w:
        df_g.to_excel(w,sheet_name='Global_COCO',index=False)
        df_d.to_excel(w,sheet_name='Por_Fecha_COCO',index=False)
        pd.DataFrame([PROTO]).to_excel(w,sheet_name='Protocolo',index=False)
        pd.DataFrame([{'limitacion':x} for x in report['notas']]).to_excel(w,sheet_name='Limitaciones',index=False)
    REP_FILE.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf8')
    print('✅ Diferencias YOLO-Mask R-CNN:',dif)
    display(df_g)


## 7. Exportar ZIP completo para auditoría reproducible

Se comprimen Excel, informes JSON, anotaciones y predicciones RLE. **No contiene los pesos de cientos de MB ni las imágenes originales.** Puedes enviarme este ZIP y procesar el resto en CPU.

In [ ]:
from google.colab import files
ZIP=EXPORT/'CERDOS_09_COCO_Armonizado_Resultados.zip'
files_to_zip=[EXCEL,REP_FILE,CONFIG_FILE,GT_FILE,YPRED_FILE,MPRED_FILE]
assert all(p.is_file() for p in files_to_zip),'Faltan archivos; no exportar resultados incompletos'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as archive:
    for file in files_to_zip:archive.write(file,arcname=file.name)
with zipfile.ZipFile(ZIP) as archive:
    assert archive.testzip() is None,'ZIP corrupto'
print('✅ ZIP:',ZIP,'| MB:',round(ZIP.stat().st_size/1024**2,2))
files.download(str(ZIP))


**Siguiente:** ejecutar `10_CERDOS_Analisis_Fecha_Incertidumbre_CPU.ipynb` **sin acelerador**. No volver a entrenar modelos para esta comparación. Una evaluación externa verdaderamente independiente exigirá imágenes de otros días/granjas o nuevas secuencias no utilizadas.